In [ ]:
%%sql
select * from edw.tbl_fact_policy_quotes limit 100

In [ ]:
%%sql
select * from edw.tbl_fact_policy_sales limit 100

In [ ]:
%%sql
select * from edw.tbl_fact_policy_renewals limit 100

In [ ]:
%%sql
select * from dlk.hfq_quotedetails limit 100

In [ ]:
%%sql
select * from dlk.genesys_session_summary 
Where conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' and direction = 'outbound' limit 100

Create Event Stream

In [ ]:
Create Table if not exists ods.EpisodeEventStream 
(
    EventId                 BigInt IDENTITY,
    EventSourceId           Int,
    EventSourceReference    Varchar(128),
    SourceQuoteReference    Varchar(128),
    SourcePolicyReference   Varchar(128),
    SourceCustomerReference Varchar(128),
    SourceSystemId          int,
    PolicyTypeGroup         varchar(50),
    EventDateTime           TIMESTAMP,
    EventDate               Date,
    EventTypeId             int,
    EventDescription        varchar(1024),
    Grain                   varchar(50)
)

## Populate step R1

In [201]:
/* Renewals and sales to be derived properly - we are picking up from DWH for now */
/* Renewals are always derived for the month of renewals -- and their sales over time across months */ 
Create or Replace Table stg.R0_MotorPoliciesEligibleForRenewals as 
select  PolicyCode, LyPolicyRenewDateAdj as RenewalDate, LYReportingSaleDate, TyPolicyCode , TYReportingSaleDate, TyPolicyRenewDateAdj, Channel, PolicyOfferNum, PolicyRetNum,
 RenEURPremOffer, TYEURGrossPremium


from    edw.tbl_fact_policy_renewals    a 
Where   a. LyPolicyTypeGroup = 'Motor' 
and     a.LYPolicyRenewDateAdj between '2026-07-01' and '2026-07-31' 
-- and     PolicyOfferNum = 1 
Group by   PolicyCode, LyPolicyRenewDateAdj, LYReportingSaleDate,  TyPolicyCode  , TYReportingSaleDate,  TyPolicyRenewDateAdj, Channel, PolicyOfferNum, PolicyRetNum,
 RenEURPremOffer, TYEURGrossPremium

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 159, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
select  count(distinct PolicyCode)
from    edw.tbl_fact_policy_renewals a      
Where   a.LYPolicyTypeGroup = 'Motor' 
and     a.LYPolicyRenewDateAdj between '2026-07-01' and '2026-07-31' 
and     PolicyOfferNum = 1 and RenEURPremInvite <> RenEURPremOffer

StatementMeta(, 3071d2ff-ce2e-4158-b222-a1abb290a4f0, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
select count(*) from stg.R0_MotorPoliciesEligibleForRenewals

StatementMeta(, 3071d2ff-ce2e-4158-b222-a1abb290a4f0, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [172]:
/* R1a */ 
Create or Replace Table stg.R1_Motor_Renewals_EmailOffered as 
select  SourcePolicyReference, min(EventDateTime) OfferedDateTime
from    ods.EventStream                             a, 
        stg.R0_MotorPoliciesEligibleForRenewals     b 
Where   EventDescription like 'Renewal Offer - Emailed Document %'
and     a.PolicyTypeGroup = 'Motor'
and     a.SourcePolicyReference = b.PolicyCode 
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by SourcePolicyReference
;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 124, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.SourcePolicyReference,
        1,
        'Motor',
        max(date_trunc('MINUTE', OfferedDateTime)),
        max(cast(OfferedDateTime as date)),
        'R1a',
        'Motor Renewal - Renewal invitation sent by EMAIL',
        'Policy'
FROM    stg.R1_Motor_Renewals_EmailOffered
;

In [174]:
/* R1a */
select count(*) from stg.R1_Motor_Renewals_EmailOffered ;
select count(*) from stg.R1_Motor_Renewals_PostOffered ;
select count(*) from stg.R1_Motor_Renewals_EmailOffered a, stg.R1_Motor_Renewals_PostOffered b where a.SourcePolicyReference = b.SourcePolicyReference;



StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 128, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [173]:
/* R1b */ 
Create or Replace Table stg.R1_Motor_Renewals_PostOffered as  
select  SourcePolicyReference, min(EventDateTime) OfferedDateTime
from    ods.EventStream a, stg.R0_MotorPoliciesEligibleForRenewals b 
Where   EventDescription like 'Renewal Offer - Document Transmitted %'
and     a.PolicyTypeGroup = 'Motor'
and     a.SourcePolicyReference = b.PolicyCode 
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by SourcePolicyReference
;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 125, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.SourcePolicyReference,
        1,
        'Motor',
        max(date_trunc('MINUTE', OfferedDateTime)),
        max(cast(OfferedDateTime as date)),
        'R1b',
        'Motor Renewal - Renewal invitation sent by EMAIL',
        'Policy'
FROM    stg.R1_Motor_Renewals_PostOffered
Group by a.SourcePolicyReference
;

In [33]:
/* R2 */ 
select  count(*), count(distinct ClientCode), count(distinct PolicyCode)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.appliedrenewals_successfulloginevent b 
Where   a.ClientCode = b.PortfolioCode 
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate
;

StatementMeta(, 280133ab-f85c-4f25-8ca8-313555c9d1fc, 37, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', 'Timestamp')),
        max(cast('Timestamp' as date)),
        'R2',
        'Motor Renewal - Customer logs in to portal',
        'Policy'
FROM    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.appliedrenewals_successfulloginevent b 
Where   a.ClientCode = b.PortfolioCode 
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate
group by a.PolicyCode
;

In [32]:
/* R2.F1 */ 
select  count(*), count(distinct ClientCode), count(distinct PolicyCode)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.appliedrenewals_failedloginevent b 
Where   a.ClientCode = b.PortfolioCode 
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate;

StatementMeta(, 280133ab-f85c-4f25-8ca8-313555c9d1fc, 36, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', 'Timestamp')),
        max(cast('Timestamp' as date)),
        'R2',
        'Motor Renewal - Customer logs in to portal',
        'Policy'
FROM    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.appliedrenewals_failedloginevent b 
Where   a.ClientCode = b.PortfolioCode 
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate
group by a.PolicyCode
;

In [38]:
/* R2.B1 */
-- logic is a bit convoluted 
-- This could be a) all people who do an MFQ who are eligible for renewals or b) renewal people who pay via MFQ c) failed login people who go to mfq 

-- Let us do a for now
Create or Replace Table stg.RenewalScvCustomerKeys as 
select  distinct scv_customer_key , PolicyCode, ClientCode 
from    ods.scv_customer_key  a, (select PolicyCode, left(PolicyCode,6) as ClientCode from stg.R0_MotorPoliciesEligibleForRenewals) b 
Where   a.SourceSystemReference = b.ClientCode 
and     a.SourceSystemId = 1 
;

StatementMeta(, c0290118-b26a-43fc-9ba8-27e7a4c1dbe0, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [177]:
/* R2.B1 */
-- logic is a bit convoluted 
-- This could be a) all people who do an MFQ who are eligible for renewals or b) renewal people who pay via MFQ c) failed login people who go to mfq 

-- Let us do a for now /* should have got the created date in ods.MFQSCVLink */ 
Create or Replace Table stg.RenewalsDoingMFQ as 
select  a.*, b.SourceSystemQuoteReference MFQQuoteQueryGuid , QuoteStartDatetime
from    stg.RenewalScvCustomerKeys a, ods.MFQSCVLink b, ods.scv_customer_key c   
Where   a.scv_customer_key = c.scv_customer_key 
and     b.RapierCustomerId = c.ChillSourceCustomerKey 
and     QuoteStartDatetime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 132, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [178]:

/* R2.B1 */
select count(distinct PolicyCode), count(distinct MFQQuoteQueryGuid)
from stg.RenewalsDoingMFQ 

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 133, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', QuoteStartDatetime)),
        max(cast(QuoteStartDatetime as date)),
        'R2.B1',
        'Motor Renewal - Online re-quote (MFQ) for another price',
        'Policy'
FROM    stg.RenewalsDoingMFQ 
group by PolicyCode
;

In [73]:
/* inverse of R2.B1.F1 -- these people who are supposed to renew bought policy from MFQ -- so out of 1616, 340 bought from MFQ -- the rest 1276 did not */
select  count(distinct PolicyCode)
from    stg.MFQ_Quote_Payments a, stg.RenewalsDoingMFQ  b 
Where   a.QuotequeryGuId = b.MFQQuotequeryGuId and PaymentType in (0,5)
limit 10 ;

/* R2.B1.F1  */
select count(distinct a.PolicyCode) 
from    stg.RenewalsDoingMFQ  a 
left join 
        (
            select  PolicyCode
            from    stg.MFQ_Quote_Payments a, stg.RenewalsDoingMFQ  b 
            Where   a.QuotequeryGuId = b.MFQQuotequeryGuId 
            and     PaymentType in (0,5)
            and     PaymentDate  between '2026-05-01 00:00:00.000' and '2026-08-31 00:00:00.000' 
            Group by PolicyCode
        ) b 
        on b.PolicyCode = a.PolicyCode
where b.PolicyCode is null 

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 19, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', QuoteStartDatetime)),
        max(cast(QuoteStartDatetime as date)),
        'R2.B1.F1',
        'Motor Renewal - New quote does not beat the renewal price, or the quote is abandoned',
        'Policy'
FROM    stg.RenewalsDoingMFQ  a 
left join 
        (
            select  PolicyCode
            from    stg.MFQ_Quote_Payments a, stg.RenewalsDoingMFQ  b 
            Where   a.QuotequeryGuId = b.MFQQuotequeryGuId 
            and     PaymentType in (0,5)
            and     PaymentDate  between '2026-05-01 00:00:00.000' and '2026-08-31 00:00:00.000' 
            Group by PolicyCode
        ) b 
        on b.PolicyCode = a.PolicyCode
where b.PolicyCode is null  
group by a.PolicyCode
;

In [50]:
/* R2.B1.F1 example -- the quoted prices are cheaper but the client went for a higher price .. why ? */
select * from edw.tbl_fact_policy_renewals where PolicyCode = 'HEGJ91004' and RenewalMonth = '2026-07-31'
-- HEGJ91004,HEGJ91,1ebb6584-559e-4486-91de-a7a80d463234

StatementMeta(, ecbbf383-6e53-431e-bbc2-85fa80649232, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 314 fields>

In [66]:
-- R3a - component 1
-- (R3a  / b might include a transfer in which case the sold policy this year may ahve a different policy code - one to check for)
select  count(*), count(distinct ClientCode), count(distinct PolicyCode)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentSuccess b 
Where   PolicyCode = PolicyCodeForRenewal
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [68]:
-- R3a - component 2
-- (R3a  / b might include a transfer in which case the sold policy this year may ahve a different policy code - one to check for)
select  count(*), count(distinct ClientCode), count(distinct PolicyCodeForRenewal)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentSuccess b 
Where   ClientCode = left(PolicyCodeForRenewal,6)
and     PolicyCode <> PolicyCodeForRenewal
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [ ]:
-- R3a - component 3
select  count(distinct PolicyCode)
from    stg.MFQ_Quote_Payments a, stg.RenewalsDoingMFQ  b 
Where   a.QuotequeryGuId = b.MFQQuotequeryGuId and PaymentType in (0,5)
limit 10 ;

In [95]:
-- R3a -- Combine them all 
Create or Replace Table stg.MotorRenewalsOnline as 
select  distinct PolicyCode, SalesSource,,RenewalStartDate
from   (select  PolicyCode, 1 SalesSource,RenewalStartDate
        from    (
                    select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
                    from stg.R0_MotorPoliciesEligibleForRenewals 
                )   a,
                dlk.AppliedRenewals_PaymentSuccess b 
        Where   PolicyCode = PolicyCodeForRenewal
        and     b.`Timestamp` > a.RenewalStartDate
        and     b.`Timestamp` < a.RenewalEndDate
        union all 
        select  PolicyCodeForRenewal, 2 ,RenewalStartDate
        from    (
                    select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
                    from stg.R0_MotorPoliciesEligibleForRenewals 
                )   a,
                dlk.AppliedRenewals_PaymentSuccess b 
        Where   ClientCode = left(PolicyCodeForRenewal,6)
        and     PolicyCode <> PolicyCodeForRenewal
        and     b.`Timestamp` > a.RenewalStartDate
        and     b.`Timestamp` < a.RenewalEndDate
        union all
        select  PolicyCode, 3 , a.EventDateTime
        from    stg.MFQ_Quote_Payments a, stg.RenewalsDoingMFQ  b 
        Where   a.QuotequeryGuId = b.MFQQuotequeryGuId and PaymentType in (0,5)
        ) x 


StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 41, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [88]:
select count(*) from stg.MotorRenewalsOnline

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 34, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', RenewalStartDate)),
        max(cast(RenewalStartDate as date)),
        'R3a',
        'Motor Renewal - Customer renews ONLINE',
        'Policy'
FROM    stg.MotorRenewalsOnline
group by PolicyCode
;

In [83]:
-- R3b - can derive from NewArchive as well as DWH - Let us derive from NewArchive first 
-- Did not work as expected -- IGNORE for now
Create or Replace table stg.NewArchiveMotorRenewalSales as 
select  ar_policy_code  
from    dlk.rbsdata2_newarchive a, edw.tbl_ref_Mapping_BAR_Premium_Type b 
Where   replace(a.ar_cli_type, ' ', '-')        = b.ar_cli_type 
and     replace(a.ar_com_type, ' ', '-')        = b.ar_com_type
and     replace(a.ar_return_sub_type, ' ', '-') = b.ar_return_sub_type 
and     replace(a.ar_com_sta, ' ', '-')         = b.ar_com_sta
and     replace(a.ar_confirmed_prov, ' ', '-')  = b.ar_confirmed_prov
and     PremiumTypeGroup = 'Renewals'
and     a.ar_posting_date between 20260501 and 20260830 
and     a.ar_pol_type = 'M1' 
Group by ar_policy_code  

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 29, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [86]:
select count(*) from stg.NewArchiveMotorRenewalSales a, stg.R0_MotorPoliciesEligibleForRenewals  b where a.ar_policy_code = b.PolicyCode 

-- This number is not usable.. So we back off into DWH derivation for now but will come back to this after this run

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 32, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [137]:
/* R3c -- unfortunately we use this for now for R3c -- not for long though */
Create or Replace Table stg.R3c_MotorRenewalsBackOffice as 
select  distinct TyPolicyCode
from    edw.tbl_fact_policy_renewals 
where   RenewalMonth = '2026-07-31' 
and     channel = 'a) IB' 
and     PolicyRetNum = 1 
and     PolicyOfferNum = 1 
and		LYPolicyTypeGroup = 'Motor' 

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 83, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [150]:
/* R3 d ?  -- WebAssist -- unfortunately we use this for now for the next set  -- not for long though */
Create or Replace Table stg.R3d_MotorRenewalsWebAssist as 
select    distinct TyPolicyCode
from    edw.tbl_fact_policy_renewals 
where   RenewalMonth = '2026-07-31' 
and     channel like '%eb %'
and     PolicyRetNum = 1 
and     PolicyOfferNum = 1 
and		LYPolicyTypeGroup = 'Motor' 

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 100, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [128]:
/* R3b.F1 Genesys Call Wait Times */ 
--  stg.A0_genesys_derived_data is already derived but remember to not use one stg from one notebook in another notebook --- that can lead to disaster.. 
-- each notebook is self contained for stg tables.. ods / dlk / prd are the dependable tables across schemas  

Create or Replace Table stg.R0_genesys_derived_data as 
select  conversationId, 
        case 
            when originatingDirection = 'inbound' then replace(ani,'tel:+353', '0') 
            when originatingDirection = 'outbound' then replace(dnis,'tel:+353', '0') 
            else null 
        end as CustomerPhoneNumber,
        originatingDirection,          
        conversationStartTime,
        conversationEndTime,
        sessionCount,
        sessionIndex,
        sessionDuration,
        sessionStartTime,
        sessionEndTime,
        queueName,
        purpose,
        transferToId,
        agentAnswered, 
        alertNoAnswer, 
        abandoned, 
        totalAcdWaitDuration, 
        totalAgentAlertDuration, 
        totalAgentHoldDuration, 
        totalAgentTalkDuration
from    dlk.genesys_session_summary
Where	conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 74, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [129]:
Create or Replace Table stg.R3bF1_genesys_inbound_call_duration_summary as 
select	a.ConversationId, min(conversationStartTime) as conversationStartTime, 
            sum(agentAnswered) as agentAnswered, sum(alertNoAnswer) alertNoAnswer, sum(abandoned) abandoned, 
            sum(totalAcdWaitDuration) CallWaitTime, sum(totalAgentAlertDuration) CallRingTime, sum(totalAgentHoldDuration) CallHoldTime, sum(totalAgentTalkDuration) CallSpokenTime
from	stg.R0_genesys_derived_data	a
where	a.queueName = 'INBOUND_Car_Renewals'  
Group by ConversationId

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 75, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [132]:
/* R3b.F1 */ 
select count(distinct b.CustomerPhoneNumber) from stg.R3bF1_genesys_inbound_call_duration_summary a, stg.R0_genesys_derived_data b where a.abandoned = 1 and a.CallWaitTime > 0 and a.ConversationId = b.ConversationId 

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 78, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [139]:
/* R3.B1 */ 
Create or Replace Table stg.R3B1_LapsedThisYearPolicy as 
select  distinct b.PolicyCode , b.RenewalDate
from    dlk.rbsdata2_policy_physical a, stg.R0_MotorPoliciesEligibleForRenewals b
Where   trim(a.pl_code) = trim(b.PolicyCode)  -- trim is important as the source tables have trailing spaces -- Urska needs to address this 
and     trim(a.pl_status) = 'L' 


StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 85, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [141]:
/* R3.B1 R3.O1 */ 
select  count(*) 
from    stg.R3B1_LapsedThisYearPolicy 


StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 87, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [160]:
-- R4.F1 

select  count(*), count(distinct ClientCode), count(distinct PolicyCode)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentFailedEvent b 
Where   ClientCode = b.PortfolioCode
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 112, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', RenewalStartDate)),
        max(cast(RenewalStartDate as date)),
        'R4.F1',
        'Motor Renewal - Payment failed',
        'Policy'
FROM    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentFailedEvent b 
Where   ClientCode = b.PortfolioCode
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate
group by a.PolicyCode
;

In [161]:
-- R4a 

select  PolicyCode, max(`Timestamp`) PaymentDateTime 
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentSuccess b 
Where   ClientCode = b.PortfolioCode
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate
and     PaymentType = 'Full'
Group by PolicyCode
;


StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 113, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 4 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', RenewalStartDate)),
        max(cast(RenewalStartDate as date)),
        'R4a',
        'Motor Renewal - Pay in full',
        'Policy'
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentSuccess b 
Where   ClientCode = b.PortfolioCode
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate
and     PaymentType = 'Full'
Group by PolicyCode
;

In [ ]:
-- R4c -- Possible loan but old finance or new loan?  

select  PolicyCode, max(`Timestamp`) PaymentDateTime 
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentSuccess b 
Where   ClientCode = b.PortfolioCode
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate
and     PaymentType = 'Instalments'
Group by PolicyCode
;


In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', RenewalStartDate)),
        max(cast(RenewalStartDate as date)),
        'R4c',
        'Motor Renewal - New monthly agreement',
        'Policy'
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_MotorPoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentSuccess b 
Where   ClientCode = b.PortfolioCode
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate
and     PaymentType = 'Instalments'
Group by PolicyCode
;


In [166]:
/* R5 */
SELECT  a.PolicyCode, min(EventDateTime) EventDateTime
FROM    stg.R0_MotorPoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription IN 
    (
        'Renewal Confirmation - Emailed Document - Suitability Statement',
        'Renewal Transfer - Emailed Document - Motor Suitability Statement',
        'New Business - Emailed Document - Suitability Statement',
        'Renewal Confirmation - Document Transmitted - Suitability Statement',
        'Renewal Transfer - Document Transmitted - Motor Suitability Statement',
        'New Business - Document Transmitted - Suitability Statement',
        'New Business - Document Transmitted - Motor Suitability Statement'
    )
Group by a.PolicyCode

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 118, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', a.EventDateTime)),
        max(cast(a.EventDateTime as date)),
        'R5',
        'Motor Renewal - Renewal processed, policy continued',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription IN 
    (
        'Renewal Confirmation - Emailed Document - Suitability Statement',
        'Renewal Transfer - Emailed Document - Motor Suitability Statement',
        'New Business - Emailed Document - Suitability Statement',
        'Renewal Confirmation - Document Transmitted - Suitability Statement',
        'Renewal Transfer - Document Transmitted - Motor Suitability Statement',
        'New Business - Document Transmitted - Suitability Statement',
        'New Business - Document Transmitted - Motor Suitability Statement'
    )
Group by a.PolicyCode
;

In [168]:
/* R6 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.R0_MotorPoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription IN 
    (
        'Renewal Confirmation - Document Transmitted - RNL Cert Issue Letter',
        'Renewal Confirmation - Document Transmitted - Certificate',
        'Renewal Transfer - Document Transmitted - RNL Cert Issue Letter',
        'Renewal Transfer - Document Transmitted - Certificate',
        'New Business - Document Transmitted - NB Cert Issue Letter',
        'New Business - Document Transmitted - Certificate',
        'Renewal Confirmation - Emailed Document - Certificate',
        'Renewal Transfer - Emailed Document - Certificate'
        'Renewal Transfer - Printed Document - RNL Cert Issue Letter'

    )

/* 
Duplicate Certificate - Document Transmission Confirmed - IV5 - Ivernia C&D B Post
*/

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 120, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', a.EventDateTime)),
        max(cast(a.EventDateTime as date)),
        'R6',
        'Motor Renewal - Cert and disc sent',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription IN 
    (
        'Renewal Confirmation - Document Transmitted - RNL Cert Issue Letter',
        'Renewal Confirmation - Document Transmitted - Certificate',
        'Renewal Transfer - Document Transmitted - RNL Cert Issue Letter',
        'Renewal Transfer - Document Transmitted - Certificate',
        'New Business - Document Transmitted - NB Cert Issue Letter',
        'New Business - Document Transmitted - Certificate',
        'Renewal Confirmation - Emailed Document - Certificate',
        'Renewal Transfer - Emailed Document - Certificate'
        'Renewal Transfer - Printed Document - RNL Cert Issue Letter'

    )
;

In [203]:
Create or Replace Table stg.R2B1_1_Base as 
SELECT  distinct b.PolicyCode, MFQQuoteQueryGuid , b.QuoteStartDatetime , RenEURPremOffer, TYEURGrossPremium
FROM    stg.R1_Motor_Renewals_EmailOffered  a, 
        stg.RenewalsDoingMFQ                b,
        stg.R0_MotorPoliciesEligibleForRenewals c 
Where   a.SourcePolicyReference = b.PolicyCode 
and     b.PolicyCode = c.PolicyCode 
and     QuoteStartDateTime > OfferedDateTime

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 161, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [205]:
/* R2.B1.1 */
select count(distinct PolicyCode) from stg.R2B1_1_Base

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 163, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', a.QuoteStartDatetime)),
        max(cast(a.QuoteStartDatetime as date)),
        'R2.B1.1',
        'Motor Renewal - MFQ quote started from the renewal price',
        'Policy'
from    stg.R2B1_1_Base

In [204]:
/* R2.B1.3*/
create or Replace table stg.R2B1_3_base as 
select a.* from stg.R2B1_1_Base a, stg.MFQInsurersQuoted   b  Where a.MFQQuoteQueryGuid = b.QuoteQueryGuid and InsurersQuoted = 'Y' ;



StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 162, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', a.QuoteStartDatetime)),
        max(cast(a.QuoteStartDatetime as date)),
        'R2.B1.3',
        'Motor Renewal - New price returned',
        'Policy'
from    stg.R2B1_3_base

In [189]:
/* R2.B1.3.F1 */
select count(distinct a.PolicyCode) from stg.R2B1_1_Base a left join (select  distinct PolicyCode from stg.R2B1_1_Base a, stg.MFQInsurersQuoted   b  Where a.MFQQuoteQueryGuid = b.QuoteQueryGuid and InsurersQuoted = 'Y') b 
on a.PolicyCode = b.PolicyCode
Where b.PolicyCode is null 



StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 147, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', a.QuoteStartDatetime)),
        max(cast(a.QuoteStartDatetime as date)),
        'R2.B1.3.F1',
        'Motor Renewal - No price returned, or the quote is abandoned',
        'Policy'
from    stg.R2B1_1_Base a 
left join (select  
           distinct PolicyCode 
           from stg.R2B1_1_Base a, 
                stg.MFQInsurersQuoted   b  
           Where a.MFQQuoteQueryGuid = b.QuoteQueryGuid and InsurersQuoted = 'Y') b 
on a.PolicyCode = b.PolicyCode
Where b.PolicyCode is null;

In [209]:
/* R2.B1.4.F1*/
-- derive MFQ prices
Create or Replace Table stg.R2B1_4_F1_MFQPrices as 
select  a.*, Rank1PremComp
from    stg.R2B1_3_base     a,
        edw.tbl_fact_mfq    b  
Where   a.MFQQuoteQueryGuid = b.QuoteQueryGuid 
;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 167, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [214]:
/* R2.B1.4.F1*/
-- Select *
select count(distinct PolicyCode) 
from stg.R2B1_4_F1_MFQPrices 
Where Rank1PremComp > RenEURPremOffer and RenEURPremOffer > 0 
     

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 172, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', a.QuoteStartDatetime)),
        max(cast(a.QuoteStartDatetime as date)),
        'R2.B1.4.F1',
        'Motor Renewal - New price is worse than the renewal offer',
        'Policy'
from    stg.R2B1_4_F1_MFQPrices 
Where   Rank1PremComp > RenEURPremOffer and RenEURPremOffer > 0 ;

In [216]:
/* R2.B1.O1*/
select  count(distinct PolicyCode) 
from    stg.R2B1_4_F1_MFQPrices  
Where   Rank1PremComp > TYEURGrossPremium
/* renewed on TYEURGrossPremium but TYEURGrossPremium is less than Rank 1 Prem Comp */ 

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 174, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', a.QuoteStartDatetime)),
        max(cast(a.QuoteStartDatetime as date)),
        'R2.B1.O1',
        'Motor Renewal - Renews on the renewal offer',
        'Policy'
from    stg.R2B1_4_F1_MFQPrices  
Where   Rank1PremComp > TYEURGrossPremium;

In [246]:
/* R2.B1.O2*/
select  count(distinct PolicyCode) 
from    stg.R2B1_4_F1_MFQPrices  
Where   RenEURPremOffer > TYEURGrossPremium

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 204, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', a.QuoteStartDatetime)),
        max(cast(a.QuoteStartDatetime as date)),
        'R2.B1.O2',
        'Motor Renewal - Renews on a new MFQ price',
        'Policy'
from    stg.R2B1_4_F1_MFQPrices  
Where   RenEURPremOffer > TYEURGrossPremium;

In [245]:
/* R2.B1.2
 */
select  a.PolicyCode 
from    stg.R0_MotorPoliciesEligibleForRenewals a, stg.R2B1_4_F1_MFQPrices  b 
Where   a.PolicyCode = b.PolicyCode 
Group by a.PolicyCode
having count(distinct Rank1PremComp) > 1 


StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 203, Finished, Available, Finished, False)

<Spark SQL result set with 173 rows and 1 fields>

In [242]:
select  count(distinct a.PolicyCode) 
from    stg.R0_MotorPoliciesEligibleForRenewals a, stg.R2B1_4_F1_MFQPrices  b 
Where   a.PolicyCode = b.PolicyCode 
and     TyReportingSaleDate is null 

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 200, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', b.QuoteStartDatetime)),
        max(cast(b.QuoteStartDatetime as date)),
        'R2.B1.O3',
        'Motor Renewal - Buys elsewhere, no renewal',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals a, stg.R2B1_4_F1_MFQPrices  b 
Where   a.PolicyCode = b.PolicyCode 
and     TyReportingSaleDate is null;

In [217]:
/* R6.B1.5 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.R0_MotorPoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription like 'Duplicate Certificate%'

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 175, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDateTime as date)),
        'R6.B1.5',
        'Motor Renewal - Replacement requested',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription like 'Duplicate Certificate%';

In [222]:
/* R4.B2.1 */
SELECT  count(distinct a.TyPolicyCode)
FROM    stg.R0_MotorPoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription in 
        (
                'New Business - Saved Document - Document Checklist SMS',
                'New Business - Saved Document - Document Checklist Email'
            )  

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 180, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDateTime as date)),
        'R4.B2.1',
        'Motor Renewal - Document request issued',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription in 
        (
                'New Business - Saved Document - Document Checklist SMS',
                'New Business - Saved Document - Document Checklist Email'
            );

In [17]:
/* R4.B2.2 */ 
select  count(distinct b.PolicyCode)
from    stg.R0_MotorPoliciesEligibleForRenewals a, dlk.MyChill_NewUploadDocumentEvents  b 
Where   a.PolicyCode = b.PolicyCode
and     `Timestamp` > '2026-05-01 00:00:00.000' 
and     `Timestamp` < '2026-09-01 00:00:00.000' 
limit 100
;

StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 18, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [227]:
/* R4.B2.2 */ 
select  count(distinct b.PolicyCode)
from    stg.R0_MotorPoliciesEligibleForRenewals a, dlk.MyChill_NewUploadDocumentEvents  b 
Where   a.TyPolicyCode = b.PolicyCode
and     `Timestamp` > '2026-05-01 00:00:00.000' 
and     `Timestamp` < '2026-09-01 00:00:00.000' 
limit 100
;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 185, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', `Timestamp`)),
        max(cast(`Timestamp` as date)),
        'R4.B2.2',
        'Motor Renewal - Customer submits documents',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals a, dlk.MyChill_NewUploadDocumentEvents  b 
Where   a.TyPolicyCode = b.PolicyCode
and     `Timestamp` > '2026-05-01 00:00:00.000' 
and     `Timestamp` < '2026-09-01 00:00:00.000' 
group by a.PolicyCode;

In [16]:
/* R4.B2.6 / R4.B2.3 */
select  isAccepted, count(distinct b.PolicyCode)
from    stg.R0_MotorPoliciesEligibleForRenewals a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.PolicyCode = b.PolicyCode
and     `Timestamp` > '2026-05-01 00:00:00.000' 
and     `Timestamp` < '2026-09-01 00:00:00.000' 
/*and     isAccepted IsRejected */
Group by isAccepted
limit 100
;

StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 17, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [15]:
/* R4.B2.6 / R4.B2.3 */
select  isAccepted, count(distinct b.PolicyCode)
from    stg.R0_MotorPoliciesEligibleForRenewals a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.TyPolicyCode = b.PolicyCode
and     `Timestamp` > '2026-05-01 00:00:00.000' 
and     `Timestamp` < '2026-09-01 00:00:00.000' 
/*and     isAccepted IsRejected */
Group by isAccepted
limit 100
;

StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 16, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', `Timestamp`)),
        max(cast(`Timestamp` as date)),
        'R4.B2.3',
        'Motor Renewal - Documents validated',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.TyPolicyCode = b.PolicyCode
and     `Timestamp` > '2026-07-01 00:00:00.000' 
and     `Timestamp` < '2026-09-01 00:00:00.000'
and     isAccepted = 'true'
group by a.PolicyCode;

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', `Timestamp`)),
        max(cast(`Timestamp` as date)),
        'R4.B2.6',
        'Motor Renewal - Documents received and validated',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.TyPolicyCode = b.PolicyCode
and     `Timestamp` > '2026-07-01 00:00:00.000' 
and     `Timestamp` < '2026-09-01 00:00:00.000'
and     isAccepted = 'true'
group by a.PolicyCode;

In [8]:
/* R4.B2 -- doc request */
SELECT  count(distinct a.PolicyCode)
FROM    stg.R0_MotorPoliciesEligibleForRenewals     a,
        ods.EventStream                             d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     EventDateTime > '2026-05-01 00:00:00.000' 
and     EventDateTime < '2026-09-01 00:00:00.000' 
and     EventDescription like '%Emailed%' 
and     EventDescription like '%Body%'  

StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [230]:
/* R4.B2.5 -- doc escalation */
SELECT  distinct a.PolicyCode
FROM    stg.R0_MotorPoliciesEligibleForRenewals     a,
        ods.EventStream                             d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     EventDateTime > '2026-05-01 00:00:00.000' 
and     EventDateTime < '2026-09-01 00:00:00.000' 
and     EventDescription like '%Chase%' 
and     EventDescription like '%Final%'  

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 188, Finished, Available, Finished, False)

<Spark SQL result set with 942 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDateTime as date)),
        'R4.B2.5',
        'Motor Renewal - Chase, escalation',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals     a,
        ods.EventStream                             d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     EventDateTime > '2026-05-01 00:00:00.000' 
and     EventDateTime < '2026-09-01 00:00:00.000' 
and     EventDescription like '%Chase%' 
and     EventDescription like '%Final%'  
group by a.PolicyCode;

In [18]:
/* R4.B2.O1 -- no doc escalation and then cancellation */
SELECT  count(distinct a.PolicyCode)
FROM    stg.R0_MotorPoliciesEligibleForRenewals                   a,
        ods.EventStream                          d 
Where   a.TYPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     EventDateTime > '2026-05-01 00:00:00.000' 
and     EventDescription in 
        (
            'Insurer Led Cancelation - Emailed Document - Reg canx email template',
            'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template'
        )
and     a.TyPolicyCode in 
        (SELECT  distinct a.TyPolicyCode
        FROM    stg.R0_MotorPoliciesEligibleForRenewals     a,
                ods.EventStream                             d 
        Where   a.TyPolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     EventDateTime > '2026-05-01 00:00:00.000' 
        and     EventDateTime < '2026-09-01 00:00:00.000' 
        and     EventDescription like '%Chase%' 
        and     EventDescription like '%Final%'
        union 
        SELECT  distinct a.PolicyCode
        FROM    stg.R0_MotorPoliciesEligibleForRenewals     a,
                ods.EventStream                             d 
        Where   a.PolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     EventDateTime > '2026-05-01 00:00:00.000' 
        and     EventDateTime < '2026-09-01 00:00:00.000' 
        and     EventDescription like '%Chase%' 
        and     EventDescription like '%Final%')  /* do we need to check Ty and Ly Policies? */ 
        

StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 19, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDateTime as date)),
        'R4.B2.O1',
        'Motor Renewal - Cancellation, non-receipt',
        'Policy'
from    stg.R0_MotorPoliciesEligibleForRenewals                   a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     EventDateTime > '2026-05-01 00:00:00.000' 
and     EventDescription in 
        (
            'Insurer Led Cancelation - Emailed Document - Reg canx email template',
            'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template'
        )
and     a.TyPolicyCode in 
        (SELECT  distinct a.TyPolicyCode
        FROM    stg.R0_MotorPoliciesEligibleForRenewals     a,
                ods.EventStream                             d 
        Where   a.TyPolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     EventDateTime > '2026-05-01 00:00:00.000' 
        and     EventDateTime < '2026-09-01 00:00:00.000' 
        and     EventDescription like '%Chase%' 
        and     EventDescription like '%Final%' ) 
group by a.PolicyCode;

In [219]:
/* R6 */
SELECT  EventDescription, count(distinct a.PolicyCode)
FROM    stg.R0_MotorPoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference /* which policy to be compared is to be managed */
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by EventDescription
Order by 2 desc 



New Business - Emailed Document - Insurance Product Information Document
New Business - Emailed Document - Schedule

Document Chase - Emailed Document Failed - Chase Final Notice Email
Document Chase - Saved Document - Chase Final Notice Email

Renewal Transfer - Emailed Document - Key Care

2C - No Claims Bonus:- Attempt - 1 - Accepted
2P - Proof of Identification:- Attempt - 1 - Rejected

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 177, Finished, Available, Finished, False)

<Spark SQL result set with 892 rows and 2 fields>

In [1]:
/* purchase confirmation email from Amber A5.1 - email confirmation -- this pulls 2890 ecs and aligns with Nikita's number


    Ewan - please add this to event stream 
        
        but policy code / client code is sparsely populated 
        
        so we may have to match to email / phone with pii and then to ods.scv and the scv key is to be pulled? */

select	*
from	dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name in 
            (
                'Motor - Renewals Chase - 10DTI',
                'Motor - Renewals Chase - 20DTI',
                'Motor - Renewals Chase - 2DTI'
            )
and     MessageType = 'EMAIL'
and     interaction_type = 'sent' 
limit 100 

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 2, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 17 fields>

In [2]:
select  *from dlk.cda_loanse

StatementMeta(, d8dc7ba4-37bb-440b-870d-46271665ae32, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 23 fields>

In [3]:
select count(*) from dlk.rbsdata2_diary

StatementMeta(, d8dc7ba4-37bb-440b-870d-46271665ae32, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
-- get chase emails from XP table
select campaign_name from ods.EXT_XtremePushResults where campaign_name like '%Chase%' 
and [timestamp] between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by campaign_name
/* EXT_XtremePushResults_Policy is at policycode level -- the above is email level */ 
/*
Arrears - Chase 1 - 1 Day Past
Arrears - Chase 2 - 7 Days ago
Arrears - Chase 3 - 14 Days Past
Arrears - Chase 4 - 21 Days Past
Arrears - Chase 5 - 28 Days Past
Home - Post Renewal &amp; Purchase - xSell 
Home - Renewals Chase - 10 DTI
Home - Renewals Chase - 10 DTI - Ex-Chrome
Home - Renewals Chase - 20DTI
Home - Renewals Chase - 20DTI - Ex-Chrome
Home - Renewals Chase - 2DTI
Home - Renewals Chase - 2DTI - Exh-Chrome
Motor - Post Renewal & Purchase - xSell
Motor - Renewals Chase - 10DTI
Motor - Renewals Chase - 20DTI
Motor - Renewals Chase - 2DTI
Renewal Chase 16 Fri
Renewal Chase 16 Mon-Thur
Renewal Chase 21 Fri
Renewal Chase 21 Mon-Thur
SurveyPurchased Auto PIF
SurveyPurchased July23
Van - Renewals Chase - 10DTI
Van - Renewals Chase - 20DTI
Van - Renewals Chase - 2DTI
*/

In [2]:
/* R1.E1
*/
  -- get Renewal emails from XP table
select  count(distinct user_id), count(*)
from    dlk.EXT_XtremePushResults_Policy a, stg.R0_MotorPoliciesEligibleForRenewals b 
where campaign_name like '%Motor - Renewals%' and a.PolicyCode = b.PolicyCode 
and     `timestamp` between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     interaction_type = 'sent' 
and     MessageType  in ( 'EMAIL', 'SMS') 




StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [3]:
/* R1.E1
*/
  -- get Renewal emails from XP table
select  count(distinct user_id), count(*)
from    dlk.EXT_XtremePushResults_Policy a, stg.R0_MotorPoliciesEligibleForRenewals b 
where campaign_name like '%This Year Lapsed%' and a.PolicyCode = b.PolicyCode 
and     `timestamp` between '2026-05-01 00:00:00.000' and '2026-08-10 00:00:00.000' 
and     interaction_type = 'sent' 
and     MessageType  in ( 'EMAIL', 'SMS') 


StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [13]:
/* R3.B1.1
*/
  -- get Renewal emails from XP table
select  count(distinct user_id), count(*)
from    dlk.EXT_XtremePushResults a, stg.R0_MotorPoliciesEligibleForRenewals b 
where campaign_name like '%Motor Renewal Lapsed%' and a.PolicyCode = b.PolicyCode 
and     `timestamp` between '2026-05-01 00:00:00.000' and '2026-08-10 00:00:00.000' 
and     interaction_type = 'sent' 
and     MessageType  in ( 'EMAIL', 'SMS')

StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [5]:
/* docs asked for A6.B1.4


 */
select  
		count(distinct PolicyCode) 
from edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-05-01' and '2026-07-31' 
and PolicyType = 'Renewals' 
and PolicyTypeGroup = 'Motor' 
and case 
        when Sign_Prop_Status				= 'O' then 1 
        when Sign_Stat_Of_Fact_Status		= 'O' then 1 
        when Comp_Fin_Agree_Status			= 'O' then 1 
        when Comp_DDM_Status				= 'O' then 1 
        when Proof_Of_NCB_Status			= 'O' then 1 
        when Trans_Proof_Of_NCB_Status		= 'O' then 1 
        when Ltr_Of_Driv_Exp_Status			= 'O' then 1 
        when Prop_Driv_Lic_Status			= 'O' then 1 
        when Nam_Driv_Lic_Status			= 'O' then 1 
        when Gap_In_Cov_Ltr_Status			= 'O' then 1 
        when `2ndCar_Cert_Status`			= 'O' then 1 
        when Doc_Ltr_Status					= 'O' then 1 
        when Engineers_Rpt_Status			= 'O' then 1 
        when NCT_Status						= 'O' then 1 
        when Veh_Lic_Cert_Status			= 'O' then 1 
        when Irish_Reg_Status				= 'O' then 1 
        when Main_Driver_Dec_Status			= 'O' then 1 
        when Soc_Dom_Pleas_Dec_Status		= 'O' then 1 
        when Comp_Car_Exper_Status			= 'O' then 1 
        when DD_Conf_Ltr_Status				= 'O' then 1 
        when Comp_Fin_Mand_Status			= 'O' then 1 
        when Val_For_Spec_Item_Status		= 'O' then 1 
        when Dri_Lic_Num_Status				= 'O' then 1 
        when Comp_Lost_Cert_Dec_Status		= 'O' then 1 
        when Orig_Cert_Status				= 'O' then 1 
        when Cancel_Req_Status				= 'O' then 1 
        when PPS_Num_Status					= 'O' then 1 
        when Afford_State_Status			= 'O' then 1 
        when Identification_Status			= 'O' then 1 
        when Digital_Journey_Status			= 'O' then 1 
        when Finance_Form_Status			= 'O' then 1 
        else 0
    End = 1 
and Campaign = 'DAY 1'

StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
/* docs asked for A6.B1.5


 */
select  
		count(distinct PolicyCode) 
from edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-05-01' and '2026-07-31' 
and PolicyType = 'Renewals' 
and PolicyTypeGroup = 'Motor' 
and case 
        when Sign_Prop_Status				= 'O' then 1 
        when Sign_Stat_Of_Fact_Status		= 'O' then 1 
        when Comp_Fin_Agree_Status			= 'O' then 1 
        when Comp_DDM_Status				= 'O' then 1 
        when Proof_Of_NCB_Status			= 'O' then 1 
        when Trans_Proof_Of_NCB_Status		= 'O' then 1 
        when Ltr_Of_Driv_Exp_Status			= 'O' then 1 
        when Prop_Driv_Lic_Status			= 'O' then 1 
        when Nam_Driv_Lic_Status			= 'O' then 1 
        when Gap_In_Cov_Ltr_Status			= 'O' then 1 
        when `2ndCar_Cert_Status`			= 'O' then 1 
        when Doc_Ltr_Status					= 'O' then 1 
        when Engineers_Rpt_Status			= 'O' then 1 
        when NCT_Status						= 'O' then 1 
        when Veh_Lic_Cert_Status			= 'O' then 1 
        when Irish_Reg_Status				= 'O' then 1 
        when Main_Driver_Dec_Status			= 'O' then 1 
        when Soc_Dom_Pleas_Dec_Status		= 'O' then 1 
        when Comp_Car_Exper_Status			= 'O' then 1 
        when DD_Conf_Ltr_Status				= 'O' then 1 
        when Comp_Fin_Mand_Status			= 'O' then 1 
        when Val_For_Spec_Item_Status		= 'O' then 1 
        when Dri_Lic_Num_Status				= 'O' then 1 
        when Comp_Lost_Cert_Dec_Status		= 'O' then 1 
        when Orig_Cert_Status				= 'O' then 1 
        when Cancel_Req_Status				= 'O' then 1 
        when PPS_Num_Status					= 'O' then 1 
        when Afford_State_Status			= 'O' then 1 
        when Identification_Status			= 'O' then 1 
        when Digital_Journey_Status			= 'O' then 1 
        when Finance_Form_Status			= 'O' then 1 
        else 0
    End = 1 
and Campaign = 'DAY 20'

StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
select * from stg.XPTempExtracts where 'Commercial'

        create or replace table tmp.lifelivepolicies as
        select  cast(a.PK_ContactID as varchar(255)) ClientCode,
                b.PK_ContactPolicyID
        from    dlk.life_tblcontacts        a,
                dlk.life_tblcontactpolicies b
        Where   a.PK_ContactID = b.FK_ContactID
        and     FK_PolicyStatusID = 1
        and     PolicyInceptionDate < '2026-08-29 00:00:00.000'
        AND     PolicyInceptionDate <> '1900-01-01 00:00:00.000'
        AND     PolicyMaturityDate >= '2026-08-28'
        AND     PolicyNumber IS NOT NULL
        AND     PolicyNumber <> ''
        AND     PolicyNumber <> 'GDPR'
        Order by 1

In [ ]:
select	top 100 * 
from	[msg].[Messages] 
where	RenderedMessage not like 'Hi, thanks for choosing us, we really appreciate it. We''ll be sending out your welcome pack shortly. Thanks, Chill Insurance.%' 
and		RenderedMessage not like 'Urgent! Your renewal is due tomorrow%'
and		RenderedMessage not like 'Urgent! Your renewal is due. %'
and		RenderedMessage not like 'Urgent! Your renewal is now due. %'
and		RenderedMessage <> 'Chill MFQ Chase'
and		RenderedMessage not like '%thank you for choosing Chill Insurance. We have issued your Cert and Disc.%'
and		RenderedMessage not like '%we have emailed important correspondence regarding your insurance policy %'
and		UTCDateAdded	between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 

Hi Lorraine, we have emailed important correspondence regarding your insurance policy to lorrainemurphy76@hotmail.com. Thanks, Chill Insurance.
Hi Ann Holohan, thank you for choosing Chill Insurance. We have issued your Cert and Disc.  You will have them within 5 days. To opt out text CHILL STOP to 50015.

In [11]:
-- get chase emails from XP table
select campaign_name from dlk.EXT_XtremePushResults 
where campaign_name like '%Lapsed%' 
and `timestamp` between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by campaign_name



StatementMeta(, cadd1be7-42e9-4a94-9106-ca1e890a6c15, 12, Finished, Available, Finished, False)

<Spark SQL result set with 23 rows and 1 fields>